In [1]:
pip install pandas pyarrow pandera

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


# Milestone 2 - Revised Data Pipeline Notebook

This notebook reads the three raw CSV files from the `data/raw` folder, validates and cleans them, keeps only the important columns for analytics use, and exports processed CSV files into the `data/processed` folder.

## Goals
- Read input files from `data/raw`
- Save cleaned files to `data/processed`
- Produce 3 processed CSV files from 3 raw CSV files
- Remove unnecessary generic columns
- Keep only the important fields needed for dashboards and analytics
- Add validation and fallback logic so the pipeline does not fail when columns are missing or messy

## 1. Import libraries

This step imports the Python libraries needed for file handling, data cleaning, schema validation, and logging. Pandas is used for transformation, while the standard library is used for path management and error handling.

In [2]:
import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

## 1. Set up folders and file paths

This step defines where the raw CSV files are stored and where the processed outputs will be saved.

The notebook expects the following structure:

- `data/raw/event_logs.csv`
- `data/raw/marketing_summary.csv`
- `data/raw/trend_report.csv`

The cleaned outputs will be saved to:

- `data/processed/`

In [3]:
BASE_DIR = Path(".")
RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "event_logs": RAW_DIR / "event_logs.csv",
    "marketing_summary": RAW_DIR / "marketing_summary.csv",
    "trend_report": RAW_DIR / "trend_report.csv",
}

## 2. Verify that the raw files exist

This step checks whether the notebook can find the three CSV files inside `data/raw`.

If a file is missing, the pipeline should stop early so the path problem can be fixed immediately.

In [4]:
for name, path in RAW_FILES.items():
    print(name, "->", path, "| exists:", path.exists())

event_logs -> data/raw/event_logs.csv | exists: True
marketing_summary -> data/raw/marketing_summary.csv | exists: True
trend_report -> data/raw/trend_report.csv | exists: True


In [5]:
missing_files = [str(path) for path in RAW_FILES.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following raw files were not found:\n" + "\n".join(missing_files)
    )
else:
    print("All raw files found successfully.")

All raw files found successfully.


## 3. Define the important columns

These are the useful business columns that should remain in the processed files.

Generic columns that are not useful for analytics can be removed during processing.

In [6]:
IMPORTANT_COLUMNS = {
    "event_logs": [
        "user_id",
        "event_type",
        "event_time",
        "product_id",
        "amount",
    ],
    "marketing_summary": [
        "date",
        "users_active",
        "total_sales",
        "new_customers",
        "report_generated",
    ],
    "trend_report": [
        "week",
        "avg_users",
        "sales_growth_rate",
    ],
}

## 4. Define expected data types and EDA-based fallback rules

This step tells the pipeline what type each important column should have.

Instead of a single hardcoded fallback (e.g. always `0.0` or `"unknown"`),
missing values are now filled using the dataset's own distribution:
- Numeric columns -> the column's **median** (or a per-group median, for
  columns where a natural grouping key exists, like `amount` by `event_type`).
- String columns -> the column's **mode** (most frequent value).
- A hardcoded constant is used **only** as a last resort, when a column is
  100% empty or entirely absent, since there's no real distribution to learn
  from in that case.

This avoids manufacturing a fake spike at 0.0 or "unknown" on any downstream
chart, which was the main critique of the previous version of this notebook.


In [7]:
EXPECTED_DTYPES = {
    "event_logs": {
        "user_id": "string",
        "event_type": "string",
        "event_time": "datetime",
        "product_id": "string",
        "amount": "float",
    },
    "marketing_summary": {
        "date": "datetime",
        "users_active": "float",
        "total_sales": "float",
        "new_customers": "float",
        "report_generated": "string",
    },
    "trend_report": {
        "week": "string",
        "avg_users": "float",
        "sales_growth_rate": "float",
    },
}

# Some NaNs are not "missing data" -- they are structurally correct zeros.
# e.g. `amount` is NaN for every non-checkout event by design (a login
# event never has a dollar amount). Format: {dataset: {column: (group_col, value_that_IS_transactional)}}
STRUCTURAL_ZERO_COLUMNS = {
    "event_logs": {},  # amount is NOT structural -- genuinely missing across all event types
}

# Columns where a per-group median is more meaningful than a single
# dataset-wide median (e.g. checkout amount varies a lot by product).
GROUP_IMPUTE_COLUMNS = {
    "event_logs": {"amount": "event_type"},
}

# Hard, fixed fallback values. These are now a LAST RESORT ONLY -- used
# when a column is completely absent from the source file, or has zero
# non-null values, so there is nothing to learn a distribution from.
HARD_FALLBACK_VALUES = {
    "string": "unknown",
    "float": 0.0,
    "datetime": "1970-01-01",
}


# L0 -- columns that identify a row. If any of these are null, the row
# can't be validated/joined downstream, so it's dropped rather than kept
# with a fabricated key.
NULL_KEY_COLUMNS = {
    "event_logs": ["user_id", "event_type", "event_time"],
    "marketing_summary": ["date"],
    "trend_report": ["week"],
}

# L1 -- PII columns to mask before anything downstream sees them.
# Salted SHA-256: the same real user_id always maps to the same masked
# value, so joins/groupbys on user_id still work, but the raw ID is not
# recoverable. TODO: move PII_SALT to an environment variable / secrets
# manager before this goes anywhere near production.
PII_SALT = "finmark_ms2_static_salt_v1"
PII_COLUMNS = {
    "event_logs": ["user_id"],
}


## 4b. L0 Validation/Dedup and L1 PII Masking

These run first, right after a file is read and its columns are standardized, and before any schema/type work:
- **L0** drops rows missing a required key column, then removes exact duplicate rows.
- **L1** replaces PII columns (`user_id`) with a salted SHA-256 hash, so nothing downstream ever sees the real identifier.

In [8]:
import hashlib


def validate_and_deduplicate(df, dataset_name):
    """L0: drop rows with a missing required key column, then remove
    exact duplicate rows. Runs before any schema/type coercion so bad
    rows never reach the processed output."""
    key_cols = [c for c in NULL_KEY_COLUMNS.get(dataset_name, []) if c in df.columns]
    before_rows = len(df)

    if key_cols:
        df = df.dropna(subset=key_cols)
    df = df.drop_duplicates()

    dropped = before_rows - len(df)
    if dropped:
        print(f"  [L0 validate] dropped {dropped} row(s) (missing key column or exact duplicate)")
    return df


def mask_pii_columns(df, dataset_name):
    """L1: replace PII columns with a salted SHA-256 hash. Deterministic
    per value, so joins/groupbys on the masked column still behave the
    same as on the original -- but the real ID is not recoverable."""
    for col in PII_COLUMNS.get(dataset_name, []):
        if col in df.columns:
            df[col] = df[col].astype(str).apply(
                lambda v: hashlib.sha256((PII_SALT + v).encode("utf-8")).hexdigest()[:16]
            )
            print(f"  [L1 mask] '{col}' hashed with salted SHA-256")
    return df


## 5. Create helper functions

These helper functions standardize column names, add missing required columns, convert data types, and keep only the columns needed for the processed outputs.

In [9]:
def standardize_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


def add_missing_columns(df, dataset_name):
    """Only fires when a column is completely absent from the source file.
    There's no distribution to learn from in that case, so the hard
    constant fallback is the honest choice here -- this is the one place
    it's still appropriate to use it.
    """
    expected_cols = IMPORTANT_COLUMNS[dataset_name]
    dtype_map = EXPECTED_DTYPES[dataset_name]

    for col in expected_cols:
        if col not in df.columns:
            expected_type = dtype_map[col]
            fallback = HARD_FALLBACK_VALUES[expected_type]
            df[col] = fallback
            print(f"  [hard fallback] '{col}' missing entirely from source -> {fallback!r}")

    return df


def eda_impute_column(df, col, dtype, dataset_name):
    """Fill missing cells in an EXISTING column using the data's own
    distribution instead of a fixed constant:
      - numeric  -> median (per-group median first, if configured)
      - string   -> mode (most frequent value)
      - datetime -> median timestamp
    Falls back to the hard constant only if the column has zero real
    values to learn a distribution from.
    """
    n_missing = df[col].isna().sum()
    if n_missing == 0:
        return df

    # Structural zeros: this NaN isn't missing, it's a true absence
    # (e.g. amount for a non-checkout event). Fill those first.
    struct_rule = STRUCTURAL_ZERO_COLUMNS.get(dataset_name, {}).get(col)
    if struct_rule:
        group_col, transactional_val = struct_rule
        if group_col in df.columns:
            is_structural_zero = df[group_col] != transactional_val
            df.loc[is_structural_zero, col] = df.loc[is_structural_zero, col].fillna(0.0)
            n_missing = df[col].isna().sum()
            if n_missing == 0:
                return df

    has_real_values = df[col].notna().sum() > 0

    if not has_real_values:
        fallback = HARD_FALLBACK_VALUES[dtype] if dtype != "datetime" else pd.to_datetime(HARD_FALLBACK_VALUES["datetime"])
        df[col] = df[col].fillna(fallback)
        print(f"  [hard fallback] '{col}' has no real values to learn from -> {fallback!r}")
        return df

    if dtype == "float":
        group_col = GROUP_IMPUTE_COLUMNS.get(dataset_name, {}).get(col)
        if group_col and group_col in df.columns:
            group_median = df.groupby(group_col)[col].transform("median")
            df[col] = df[col].fillna(group_median)
        overall_median = df[col].median()
        remaining = df[col].isna().sum()
        if remaining:
            df[col] = df[col].fillna(overall_median)
        print(f"  [EDA median] '{col}': imputed {n_missing} values"
              f"{f' (grouped by {group_col}, ' if group_col else ' ('}overall median={overall_median:.2f})")

    elif dtype == "string":
        mode_val = df[col].mode(dropna=True).iloc[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  [EDA mode] '{col}': imputed {n_missing} values -> '{mode_val}'")

    elif dtype == "datetime":
        median_date = df[col].median()
        df[col] = df[col].fillna(median_date)
        print(f"  [EDA median date] '{col}': imputed {n_missing} values -> {median_date}")

    return df


def coerce_column_types(df, dataset_name):
    dtype_map = EXPECTED_DTYPES[dataset_name]

    for col, dtype in dtype_map.items():
        if dtype == "datetime":
            df[col] = pd.to_datetime(df[col], errors="coerce")
        elif dtype == "float":
            df[col] = pd.to_numeric(df[col], errors="coerce")
        elif dtype == "string":
            df[col] = df[col].astype("string")
            df[col] = df[col].replace("<NA>", pd.NA)

        df = eda_impute_column(df, col, dtype, dataset_name)

    return df


def keep_important_columns(df, dataset_name):
    return df[IMPORTANT_COLUMNS[dataset_name]].copy()


## 6. Create the main processing function

This function performs the core pipeline logic for one dataset:
1. Read the raw CSV file
2. Standardize column names
3. Add missing required columns
4. Convert columns into expected data types
5. Keep only the important columns
6. Remove duplicate rows
7. Save the cleaned file into `data/processed`

In [10]:
def process_dataset(dataset_name, file_path):
    print(f"\n=== Processing: {dataset_name} ===")

    df = pd.read_csv(file_path)
    original_shape = df.shape

    df = standardize_columns(df)
    df = validate_and_deduplicate(df, dataset_name)  # L0
    df = mask_pii_columns(df, dataset_name)          # L1
    df = add_missing_columns(df, dataset_name)       # L2
    df = coerce_column_types(df, dataset_name)       # L2
    df = keep_important_columns(df, dataset_name)
    df = df.drop_duplicates()

    output_file = OUTPUT_DIR / f"{dataset_name}_cleaned.csv"
    df.to_csv(output_file, index=False)

    summary = {
        "dataset": dataset_name,
        "input_file": str(file_path),
        "output_file": str(output_file),
        "original_rows": int(original_shape[0]),
        "original_columns": int(original_shape[1]),
        "cleaned_rows": int(df.shape[0]),
        "cleaned_columns": int(df.shape[1]),
    }

    print("Saved to:", output_file)
    print("Original shape:", original_shape)
    print("Cleaned shape:", df.shape)

    return df, summary

## 7. Run the pipeline on all three datasets

Each file is cleaned, validated, reduced to important columns, and saved as a processed CSV file in `data/processed`.

In [11]:
processed_data = {}
processing_summaries = []

for dataset_name, file_path in RAW_FILES.items():
    try:
        df_clean, summary = process_dataset(dataset_name, file_path)
        summary["status"] = "success"
        processed_data[dataset_name] = df_clean
        processing_summaries.append(summary)
    except Exception as e:
        # A single bad/corrupted file should not stop the other datasets
        # from being processed -- log it and move on to the next one.
        print(f"  [ERROR] Failed to process '{dataset_name}': {e}")
        processing_summaries.append({
            "dataset": dataset_name,
            "input_file": str(file_path),
            "output_file": None,
            "status": "failed",
            "error": str(e),
        })



=== Processing: event_logs ===
  [L1 mask] 'user_id' hashed with salted SHA-256
  [EDA median] 'amount': imputed 1016 values (grouped by event_type, overall median=1681.26)
Saved to: data/processed/event_logs_cleaned.csv
Original shape: (2000, 50)
Cleaned shape: (2000, 5)

=== Processing: marketing_summary ===
Saved to: data/processed/marketing_summary_cleaned.csv
Original shape: (100, 50)
Cleaned shape: (100, 5)

=== Processing: trend_report ===
Saved to: data/processed/trend_report_cleaned.csv
Original shape: (20, 50)
Cleaned shape: (20, 3)


## 8. Review the processing summary

This summary shows the input location, output location, and shape changes for each dataset.

In [12]:
summary_df = pd.DataFrame(processing_summaries)
summary_df

,dataset,input_file,output_file,original_rows,original_columns,cleaned_rows,cleaned_columns,status
0,event_logs,data/raw/event_logs.csv,data/processed/event_logs_cleaned.csv,2000,50,2000,5,success
1,marketing_summary,data/raw/marketing_summary.csv,data/processed/marketing_summary_cleaned.csv,100,50,100,5,success
2,trend_report,data/raw/trend_report.csv,data/processed/trend_report_cleaned.csv,20,50,20,3,success


## 9. Preview the cleaned datasets

This step displays the first few rows of each cleaned dataset so the output can be inspected quickly.

In [13]:
for dataset_name, df in processed_data.items():
    print(f"\n--- {dataset_name.upper()} ---")
    display(df.head())


--- EVENT_LOGS ---


,user_id,event_type,event_time,product_id,amount
0,0b79b857c3765b2e,checkout,2023-06-03 04:13:00,P010,1686.795
1,6e542f6d1814a828,wishlist_add,2023-06-03 05:08:00,P020,2900.630
2,9409c82689e6d63d,profile_update,2023-06-05 06:22:00,P028,1664.010
3,43cb71001c80e2d9,page_view,2023-06-06 03:45:00,P001,1643.720
4,d0ae53d06763a811,wishlist_add,2023-06-03 12:38:00,P015,1728.270



--- MARKETING_SUMMARY ---


,date,users_active,total_sales,new_customers,report_generated
0,2023-06-01,179,81287.31,9,2023-06-01 16:00
1,2023-06-02,67,74771.99,5,2023-06-02 16:00
2,2023-06-03,369,84809.74,11,2023-06-03 16:00
3,2023-06-04,94,61212.30,3,2023-06-04 16:00
4,2023-06-05,402,80911.49,10,2023-06-05 16:00



--- TREND_REPORT ---


,week,avg_users,sales_growth_rate
0,2023-W21,328,-0.003
1,2023-W22,280,0.088
2,2023-W23,130,0.073
3,2023-W24,291,0.077
4,2023-W25,200,-0.003


## 10. Save a pipeline run report

This creates a JSON report that records what happened during the run.

The report is useful for documentation and debugging.

In [14]:
run_report = {
    "run_timestamp": datetime.now().isoformat(),
    "raw_directory": str(RAW_DIR),
    "processed_directory": str(OUTPUT_DIR),
    "datasets": processing_summaries,
}

report_path = OUTPUT_DIR / "pipeline_run_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(run_report, f, indent=4)

print("Pipeline run report saved to:", report_path)

Pipeline run report saved to: data/processed/pipeline_run_report.json


## 11. Final notes

This notebook completes the start of the data pipeline by:
- reading raw files from `data/raw`
- cleaning and validating the data
- keeping only the important analytics columns
- **imputing missing values using each column's own distribution (EDA
  median/mode), instead of a single hardcoded fallback constant**
- saving processed outputs to `data/processed`

A hardcoded fallback (`0.0` / `"unknown"` / `1970-01-01`) is now used only
as a last resort, when a column is completely absent from the source file
or has zero real values to learn a distribution from. This keeps the
cleaned data's shape closer to reality and avoids a fake spike at 0.0 or
"unknown" showing up on any downstream chart.

This can be extended later with:
- stricter schema validation using Pandera
- null percentage checks with per-column EDA reports before choosing an
  imputation strategy


In [15]:
print("Generated files:")
for file in OUTPUT_DIR.glob("*.csv"):
    print("-", file.name)

Generated files:
- event_logs_cleaned.csv
- trend_report_cleaned.csv
- marketing_summary_cleaned.csv
